In [45]:
import re
import math
import numpy as np
import pandas as pd
from pathlib import Path

# === User settings ===
# file_path = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\best_params\DOE\design[0.6, 0.6, 60]\SIM_shrinkage\[0.55, 0.43, 58.42]_N1.2536_F0_sub0.6\polar-10.txt"
file_path = r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\best_params\Optimize_Regression\[0.45, 0.57, 79.91]\SIM_fillet\[0.45, 0.57, 79.91]_N1.2536_F2_53_34_sub0.6\polar-70.txt"
target_power = 0.28  # target total power (W)
csv_out_path = (
    None  # specify output CSV path if desired, e.g., r"C:\temp\polar50_detail.csv"
)

# === Read file ===
theta_list, trans_list, refl_list = [], [], []
after_two_col = False

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        nums = re.findall(r"[-+]?\d+(?:\.\d+)?", line)
        if len(nums) == 2:
            after_two_col = True
            continue
        if not after_two_col:
            continue
        if len(nums) == 3:
            theta_list.append(float(nums[0]))
            trans_list.append(float(nums[1]))
            refl_list.append(float(nums[2]))

theta_deg = np.array(theta_list)
theta_rad = np.deg2rad(theta_deg)
I_trans = np.array(trans_list)
I_refl = np.array(refl_list)
I_sum = I_trans + I_refl

# === Integration A = ∫ (I_trans + I_refl) sinθ dθ ===
A = np.trapezoid(I_sum * np.sin(theta_rad), theta_rad)

# Derive Δφ (rad) from target power
delta_phi_rad = target_power / A
delta_phi_deg =  math.degrees(delta_phi_rad)

# === Compute solid angle (ΔΩ) and power contribution per bin ===
dtheta = np.deg2rad(np.diff(theta_deg).mean())
sin_theta = np.sin(theta_rad)
dOmega = sin_theta * dtheta * delta_phi_rad

P_bin_trans = I_trans * dOmega
P_bin_refl = I_refl * dOmega
P_bin_sum = P_bin_trans + P_bin_refl

P_trans = P_bin_trans.sum()
P_refl = P_bin_refl.sum()
P_total = P_bin_sum.sum()

# === Build output table ===
df = pd.DataFrame(
    {
        "theta_deg": theta_deg,
        "I_trans_W_per_sr": I_trans,
        "I_refl_W_per_sr": I_refl,
        "I_sum_W_per_sr": I_sum,
        "sin_theta": sin_theta,
        "dtheta_rad": np.full_like(theta_rad, dtheta),
        "delta_phi_rad": np.full_like(theta_rad, delta_phi_rad),
        "dOmega_sr": dOmega,
        "P_bin_trans_W": P_bin_trans,
        "P_bin_refl_W": P_bin_refl,
        "P_bin_sum_W": P_bin_sum,
    }
)

if csv_out_path is None:
    csv_out_path = str(Path(file_path).with_name("polar_I_dOmega_breakdown.csv"))

df.to_csv(csv_out_path, index=False)

# === Print results ===
print("\n=== Integration Results ===")
print(f"A = ∫(I_trans+I_refl)sinθ dθ = {A:.9f}")
print(f"Δφ = {delta_phi_rad:.9f} rad = {delta_phi_deg:.6f}°")
print(f"Transmitted power = {P_trans:.9f} W")
print(f"Reflected power   = {P_refl:.9f} W")
print(f"Total power       = {P_total:.9f} W (target {target_power} W)")
print(f"CSV file saved to: {csv_out_path}")

# Display first few rows for verification
print("\n--- First few rows ---")
print(df.head(10).to_string(index=False))


=== Integration Results ===
A = ∫(I_trans+I_refl)sinθ dθ = 6.776949661
Δφ = 0.041316524 rad = 2.367262°
Transmitted power = 0.205824606 W
Reflected power   = 0.074185839 W
Total power       = 0.280010445 W (target 0.28 W)
CSV file saved to: C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\best_params\Optimize_Regression\[0.45, 0.57, 79.91]\SIM_fillet\[0.45, 0.57, 79.91]_N1.2536_F2_53_34_sub0.6\polar_I_dOmega_breakdown.csv

--- First few rows ---
 theta_deg  I_trans_W_per_sr  I_refl_W_per_sr  I_sum_W_per_sr  sin_theta  dtheta_rad  delta_phi_rad  dOmega_sr  P_bin_trans_W  P_bin_refl_W  P_bin_sum_W
     0.000          0.000000         0.000000        0.000000   0.000000     0.00561       0.041317   0.000000   0.000000e+00  0.000000e+00     0.000000
     0.321          0.000000         0.000000        0.000000   0.005602     0.00561       0.041317   0.000001   0.000000e+00  0.000000e+00     0.000000
     0.643          0.155943         0.308634        0.464577   0.011222     0.00561 

In [ ]:
import re
import math
import numpy as np
import pandas as pd
from pathlib import Path

# === User settings ===
folder = Path(
    r"C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\best_params\Optimize_Regression\[0.45, 0.57, 79.91]\SIM_fillet\[0.45, 0.57, 79.91]_N1.2536_F2_53_34_sub0.6_new"
)
target_power = 0.3 # W
save_per_file_breakdown = True 
summary_csv_path = folder / "polar_summary.csv"


def read_polar_file(path: Path):
    theta_list, trans_list, refl_list = [], [], []
    after_two_col = False
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            nums = re.findall(r"[-+]?\d+(?:\.\d+)?", line)
            if len(nums) == 2:
                after_two_col = True
                continue
            if not after_two_col:
                continue
            if len(nums) == 3:
                theta_list.append(float(nums[0]))
                trans_list.append(float(nums[1]))
                refl_list.append(float(nums[2]))
    theta_deg = np.asarray(theta_list, dtype=float)
    I_trans = np.asarray(trans_list, dtype=float)
    I_refl = np.asarray(refl_list, dtype=float)

    order = np.argsort(theta_deg)
    theta_deg = theta_deg[order]
    I_trans = I_trans[order]
    I_refl = I_refl[order]
    return theta_deg, I_trans, I_refl


def midpoint_bin_widths(theta_rad: np.ndarray):
    
    if theta_rad.size == 0:
        return theta_rad
    edges = np.empty(theta_rad.size + 1, dtype=float)

    if theta_rad.size >= 2:
        edges[1:-1] = 0.5 * (theta_rad[1:] + theta_rad[:-1])
        left_gap = theta_rad[1] - theta_rad[0]
        right_gap = theta_rad[-1] - theta_rad[-2]
    else:
        edges[1:-1] = np.nan
        left_gap = right_gap = 0.0
    edges[0] = max(0.0, theta_rad[0] - 0.5 * left_gap)
    edges[-1] = min(math.pi, theta_rad[-1] + 0.5 * right_gap)
    dtheta_i = np.diff(edges)
    return dtheta_i


def process_one_file(path: Path, target_power: float, save_breakdown: bool = True):
    theta_deg, I_trans, I_refl = read_polar_file(path)
    if theta_deg.size == 0:
        return None  
    theta_rad = np.deg2rad(theta_deg)
    I_sum = I_trans + I_refl

    # A = ∫ (I_trans + I_refl) sinθ dθ
    A = np.trapezoid(I_sum * np.sin(theta_rad), theta_rad)

    # Δφ (rad) from target power
    angle_target = {
        10: 0.29148, 20: 0.2876, 30: 0.28616, 40: 0.29145,
        50: 0.28646, 60: 0.2923, 70: 0.28101, 80: 0.2699,
    }
    
    P_target = angle_target[int(re.findall(r"polar-(\d+)", path.name)[0])]
    delta_phi_rad = P_target / A
    delta_phi_deg = math.degrees(delta_phi_rad)

    # Δθ_i + dΩ_i
    dtheta_i = midpoint_bin_widths(theta_rad)
    sin_theta = np.sin(theta_rad)
    dOmega = sin_theta * dtheta_i * delta_phi_rad

    # bin power
    P_bin_trans = I_trans * dOmega
    P_bin_refl = I_refl * dOmega
    P_bin_sum = P_bin_trans + P_bin_refl

    P_trans = P_bin_trans.sum()
    P_refl = P_bin_refl.sum()
    P_total = P_bin_sum.sum()

    if save_breakdown:
        df = pd.DataFrame(
            {
                "theta_deg": theta_deg,
                "I_trans_W_per_sr": I_trans,
                "I_refl_W_per_sr": I_refl,
                "I_sum_W_per_sr": I_sum,
                "sin_theta": sin_theta,
                "dtheta_rad": dtheta_i,
                "delta_phi_rad": np.full_like(theta_rad, delta_phi_rad),
                "dOmega_sr": dOmega,
                "P_bin_trans_W": P_bin_trans,
                "P_bin_refl_W": P_bin_refl,
                "P_bin_sum_W": P_bin_sum,
            }
        )
        out_path = path.with_name(path.stem + "_breakdown.csv")
        df.to_csv(out_path, index=False)

    return {
        "file": path.name,
        "A_int": A,
        "delta_phi_rad": delta_phi_rad,
        "delta_phi_deg": delta_phi_deg,
        "P_trans_W": P_trans,
        "P_refl_W": P_refl,
        "P_total_W": P_total,
        "target_W": target_power,
        "abs_err_W": P_total - target_power,
        "rel_err_%": (
            (P_total / target_power - 1.0) * 100.0 if target_power != 0 else np.nan
        ),
        "n_samples": int(theta_deg.size),
        "theta_min_deg": float(theta_deg.min()),
        "theta_max_deg": float(theta_deg.max()),
    }


def main():
    files = sorted(folder.glob("polar-*.txt"))
    if not files:
        print(f"No files matched in folder:\n{folder}")
        return
    rows = []
    for fp in files:
        try:
            res = process_one_file(
                fp, target_power, save_breakdown=save_per_file_breakdown
            )
            if res is not None:
                rows.append(res)
                print(
                    f"Processed {fp.name}: Δφ={res['delta_phi_rad']:.6f} rad "
                    f"({res['delta_phi_deg']:.3f}°) P_trans={res['P_trans_W']:.6f} W, P_refl={res['P_refl_W']:.6f} W"
                )
            else:
                print(f"Skipped empty file: {fp.name}")
        except Exception as e:
            print(f"Error processing {fp.name}: {e}")
    if rows:
        df_sum = pd.DataFrame(rows)
        df_sum.to_csv(summary_csv_path, index=False)
        print("\n=== Summary saved ===")
        print(summary_csv_path)

if __name__ == "__main__":
    main()

Processed polar-10.txt: Δφ=0.022484 rad (1.288°) P_trans=0.290474 W, P_refl=0.001006 W
Processed polar-20.txt: Δφ=0.023776 rad (1.362°) P_trans=0.287306 W, P_refl=0.000294 W
Processed polar-30.txt: Δφ=0.024724 rad (1.417°) P_trans=0.285957 W, P_refl=0.000203 W
Processed polar-40.txt: Δφ=0.036422 rad (2.087°) P_trans=0.278123 W, P_refl=0.013327 W
Processed polar-50.txt: Δφ=0.034440 rad (1.973°) P_trans=0.239051 W, P_refl=0.047409 W
Processed polar-60.txt: Δφ=0.018505 rad (1.060°) P_trans=0.292190 W, P_refl=0.000110 W
Processed polar-70.txt: Δφ=0.043285 rad (2.480°) P_trans=0.201151 W, P_refl=0.079859 W
Processed polar-80.txt: Δφ=0.027666 rad (1.585°) P_trans=0.254847 W, P_refl=0.015053 W

=== Summary saved ===
C:\Users\cchih\Desktop\NTHU\MasterThesis\research_log\best_params\Optimize_Regression\[0.45, 0.57, 79.91]\SIM_fillet\[0.45, 0.57, 79.91]_N1.2536_F2_53_34_sub0.6_new\polar_summary.csv
